# PE_V2X_Reliability — Rapport de projet (Schéma A)

Ce rapport présente la motivation, la modélisation des trois scénarios, le protocole de métriques, la reproductibilité, et une discussion figure par figure (Fig01–Fig07). Les figures sont affichées en PNG rendus à partir des PDF de référence.

**Date:** 2026-03-01

## 1 Motivation et définition du problème

L’exigence centrale des messages de sécurité V2X n’est pas “le débit maximal” ni “le délai moyen minimal”, mais la capacité à satisfaire durablement des contraintes de **fiabilité** et de **ponctualité** nécessaires à la perception/décision coopérative, sous **topologie très dynamique**, **environnements de propagation complexes** et **contention sur un canal radio partagé**. Sur route, les environnements dégradés (obstruction urbaine, tunnel) et la congestion aux heures de pointe ne sont pas des “cas extrêmes” rares ; ce sont des limites opérationnelles courantes. Il est donc justifié de construire un cadre d’évaluation **interprétable, reproductible et extensible** pour “la fiabilité et la ponctualité V2X en environnement dégradé”.

Du point de vue applicatif, au moins deux modes d’échec doivent être distingués, car leurs impacts sont différents :

1) **Échec de fiabilité (non‑livraison)** : le message n’arrive pas, la PDR diminue.  
2) **Échec de ponctualité (livraison tardive)** : le message arrive après la deadline ; pour une décision coopérative il est équivalent à un échec, i.e. un “échec caché”.

Ainsi, une métrique unique (délai moyen ou PDR seule) ne suffit pas : la moyenne masque le **risque de queue**, et sous deadline, l’épaississement de queue se traduit par plus de paquets “late”, réduisant le succès effectif. Pour éviter des conclusions “sans attribution”, ce projet adopte une modélisation système explicable et rapporte, sous protocole unifié :

- **Fiabilité selon la distance** : PDR vs distance ;  
- **Distribution des délais et risque de queue** : CDF, p95/p99 ;  
- **Succès effectif sous deadline** : `success_phy` (succès PHY), `late` (dépassement), `success` (succès à temps).

Le projet contraint ensuite l’environnement dégradé à trois scénarios représentatifs (RefPlus / UrbMaskPlus / TunnelPlus) et introduit une comparaison stricte **NoCong / Cong**. Tout le reste est figé ; seul un switch congestion introduit contention/collision. Deux questions de conception sont alors adressées :

- La congestion devient-elle le goulot d’étranglement dominant, compressant les différences d’environnement ?  
- Avec `ret=0/1/2` comme levier principal, comment se fait le compromis entre gains (fiabilité) et coûts (queue plus épaisse, late plus élevé) selon scénario/régime ?

---

## 2 Contenu réalisé et workflow global

Le livrable est une **chaîne d’évaluation système** : génération d’entrées → simulation événementielle → agrégation → figures, figée par un **contrat de sortie run_id + snapshots de commandes**. L’objectif n’est pas “un jeu de résultats”, mais une ossature réutilisable conservant le même protocole de métriques et la même chaîne de preuves pour des extensions futures (autres environnements, RSU, stratégies, trajectoires réelles).

### 2.1 Génération d’entrées : scénarios et trafic explicites/auditables

- **Mobilité** : CSV de trajectoires généré par géométrie RefPlus + modèle IDM + phases de feux + mécanismes cross/turn.  
- **Bâtiments (UrbMask)** : blocs rectangulaires, CSV bâtiments (empreintes, hauteurs, motifs spatiaux).  
- **Tunnel** : intervalle et paramètres de dégradation, CSV de configuration tunnel.

### 2.2 Simulation événementielle : échantillons paquet au niveau des événements

Pour chaque événement de message, la simulation produit :

- **Géométrie/état** : distance, position médiane, états de scénario (NLOS, inside tunnel) ;  
- **Résultats** : `success_phy`, `delay_ms`, `late`, `success` ;  
- **Champs de preuve** : obstruction (`d_min`/`b`), position tunnel (`tunnel_u`), proxys congestion (`n_cs`/CBR/`p_col`/`cong_delay_ms`).

### 2.3 Agrégation : compression des raw en tables lisibles

Les logs raw peuvent être volumineux ; l’évaluation est alignée sur des tables CSV :

- **summary_metrics** : PDR et quantiles par bins de distance, late ratio, moyennes des preuves ;  
- **position_heterogeneity** : hétérogénéité UrbMask à distance fixée (bins `mid_x`) ;  
- **tunnel_segments** : segmentation Tunnel (bins `tunnel_u`) ;

### 2.4 Figures : figures finales vs transparence

- **Fig01–Fig07** : figures finales (protocole unifié, densité élevée).  
- **F1/F2/F3** : figures de transparence et d’exploration ; elles ne remplacent pas les figures finales.

---

## 3 Importance, valeur et contributions

Cette section suit une structure “valeur–contribution–vérifiabilité”, alignée sur le travail réel sans exagération.

### 3.1 Valeur

**(1) Définition de succès alignée sur l’application : modéliser explicitement le ‘late’**  
“Reçu trop tard” équivaut à un échec pour la décision coopérative. En séparant `success_phy` et `success` et en journalisant `late`, les conclusions deviennent pertinentes pour les contraintes de sécurité.

**(2) Phénomènes mécanisés : hétérogénéité et segmentation quantifiables**  
Le canyon urbain n’est pas seulement “une moyenne plus faible”, mais une hétérogénéité à distance identique ; le tunnel est une dégradation segmentée et une queue plus épaisse. Tables dédiées + champs de preuve rendent ces phénomènes auditables.

**(3) Compromis expliquables sous congestion**  
Le gain de ret s’accompagne souvent d’un coût de queue ; sous congestion, l’augmentation du succès PHY peut être partiellement compensée par davantage de late. Fig03 + Fig04 rendent ce compromis mesurable.

**(4) Reproductibilité forte**  
Contrat run_id + snapshots + tables + figures permettent une vérification indépendante de l’arborescence de l’auteur.

### 3.2 Contributions

- Construction normalisée des trois scénarios avec paramètres explicites et champs de preuve ;  
- Chaîne de proxys congestion + ablation NoCong/Cong ;  
- Évaluation unifiée de `ret=0/1/2` sous les deux régimes ;  
- Chaîne “figure–table–champ” soutenant des narratives mécanistes (Fig03–Fig06).

### 3.3 Vérification rapide

1) Lire Fig01–Fig07 ;  
2) Vérifier `run_commands` ;  
3) Contrôler les tables ;  
4) Explorer via F1/F2/F3 si nécessaire.

---

## 4 Construction des scénarios (détaillée, avec équations)

Le chapitre suit un style “modèle système / réglages de scénarios”. Il s’agit d’un cadre d’évaluation système explorable (pas un ray tracing complet, ni une reproduction full-stack).

### 4.0 Conventions unifiées : coordonnées, temps discret, point médian

- Coordonnées planes : corridor selon \(x\), latéral \(y\).  
- Pas de temps \(\Delta t\) ; messages à fréquence \(f_m\).  
- Lien Tx/Rx :

$$
d=\|\mathbf{p}_{tx}-\mathbf{p}_{rx}\|_2,\qquad 
\mathbf{p}_{mid}=\frac{\mathbf{p}_{tx}+\mathbf{p}_{rx}}{2},\qquad
x_{mid}=\mathbf{p}_{mid,x}.
$$

### 4.1 RefPlus : baseline contrôlée (géométrie + IDM + feux + cross/turn)

$$
y_c(x)=A\sin\!\Big(2\pi\,\frac{x-x_0}{x_1-x_0}\Big)\cdot \mathbb{I}_{[x_0,x_1]}(x).
$$

IDM :

$$
\dot v = a\Big[1-\Big(\frac{v}{v_0}\Big)^{\delta}-\Big(\frac{s^{\ast}(v,\Delta v)}{s}\Big)^2\Big],
\qquad
s^{\ast}(v,\Delta v)=s_0+vT+\frac{v\Delta v}{2\sqrt{ab}}.
$$

Feux (indicateur) :

$$
\phi_{main}(t)=\mathbb{I}\big(\ (t+\tau)\bmod C\in[0,G)\ \big).
$$

### 4.2 UrbMaskPlus : obstruction urbaine

$$
d_{min}=\min_{B_k\in\mathcal{B}} \ \mathrm{dist}\big(\overline{\mathbf{p}_{tx}\mathbf{p}_{rx}},\partial B_k\big),
\qquad
b=\exp\!\left(-\Big(\frac{d_{min}}{T}\Big)^2\right),
$$
$$
p_{succ}(d,b)=(1-b)\,p_{LOS}(d)+b\,p_{NLOS}(d),
\qquad
G_{refl}(d_{min},b)=G_{max}\exp(-d_{min}/d_0)\,b,
$$
$$
\mathcal{S}_j=\{\,\text{links}: d\in[d_1,d_2],\ x_{mid}\in I_j\,\}.
$$

### 4.3 TunnelPlus : dégradation segmentée

$$
u=\mathrm{clip}\Big(\frac{x_{mid}-x_0}{x_1-x_0},0,1\Big),
\qquad
\mathrm{bell}(u)=\sin^2(\pi u),\quad \mathrm{bell}_\gamma(u)=\mathrm{bell}(u)^\gamma,
$$
$$
D = D_0 + b_{tun}\cdot D_{extra},\qquad D_{extra}\sim \mathrm{Exp}(\lambda),
\qquad
\mathcal{T}_j=\{\,\text{links}: d\in[d_1,d_2],\ u\in J_j\,\}.
$$

### 4.4 Champs de preuve

- UrbMask : `d_min`, `b`, `g_refl` ⇔ hétérogénéité (Fig05)  
- Tunnel : `tunnel_u` ⇔ segmentation (Fig06)  
- Cong : `n_cs`, CBR, `p_col`, `cong_delay_ms` ⇔ décomposition (Fig03/04)

---

## 5 Modélisation et métriques

Avec \(S=S_{phy}(1-L)\) :

$$
\mathrm{PDR}\approx \mathrm{PDR}_{phy}\cdot (1-\mathrm{LateRatio}_{phy}).
$$

Expressions conceptuelles :

$$
\mathbb{P}(S_{phy}=1)=1-\prod_{k=0}^{ret}(1-p_k),
\qquad
D = D_{base} + D_{prop} + D_{queue} + D_{cong} + D_{retry}.
$$

Proxys congestion :

$$
\mathrm{CBR}=\min\big(1,\ (n_{cs}-1)\lambda\tau_{air}\big),
\qquad
p_{col}=1-\exp(-k\,\mathrm{CBR}),
\qquad
D_{cong}\sim \mathrm{Exp}(\beta(\mathrm{CBR})).
$$

---

## 6 Reproductibilité et audit (formulation formelle)

On note `$ROOT` la racine du dépôt. Les sorties suivent `runs/<run_id>/` et incluent `run_commands.txt`. La différence NoCong/Cong est limitée à un switch congestion pour l’attribution. Reproduction recommandée : smoke → small → S20.

# Décomposition et discussion des figures (Raw-derived, S20)

## 1. Source des données et workflow statistique (raw → cache → figures)

Ces résultats reposent sur deux expériences finales strictement contrôlées : **A_Final_NoCong_S20** et **A_Final_Cong_S20**. Leur configuration commune est identique : scénarios **Ref / UrbMask / Tunnel**, stratégies **ret=0/1/2**, seeds **1–20**, statistiques centrées sur **0–200 m**. La seule différence est l’activation du mécanisme de congestion/compétition (NoCong vs Cong).

Afin d’éviter que des “tables trop clairsemées” ne produisent des courbes peu fiables, les figures finales ne sont pas tracées directement depuis les tables. Le pipeline est en deux étapes :
(1) agrégation des raw par bins de distance sous forme de comptages et d’histogrammes nécessaires aux quantiles ;
(2) cache en `.mat`, puis génération des 7 figures finales sous MATLAB.
Les fichiers de cache indiquent les résolutions : **distance bin=2 m**, **delay histogram bin=0.25 ms**, bande **band=80–100 m**.

---

## 2. Définitions des métriques et niveaux de “succès”

Sous deadline, “succès” a au moins deux niveaux : **succès PHY** et **succès à temps**. Trois métriques complémentaires :

1. **PDR à temps**
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\frac{N*{\mathrm{success}}(d)}{N_{\mathrm{total}}(d)}
$$

2. **Taux de succès PHY**
$$
\mathrm{PDR}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{success_phy}}(d)}{N_{\mathrm{total}}(d)}
$$

3. **Taux late parmi les succès PHY**
$$
\mathrm{late_ratio}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{late}}(d)}{N_{\mathrm{success_phy}}(d)}
$$

Identité clé :
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\mathrm{PDR}*{\mathrm{phy}}(d)\cdot\left(1-\mathrm{late_ratio}_{\mathrm{phy}}(d)\right)
$$

---

# Fig01–Fig07 décomposition figure par figure

## Fig01 : PDR vs distance (dist≤200 m ; NoCong plein, Cong pointillé)

![](assets/final_figures_A_preview/Fig01_PDR_Focus.png)

Ancrages Fig07 (dist≤200 m) :

NoCong :
* Ref : 0.439 → 0.537 → 0.582
* UrbMask : 0.462 → 0.584 → 0.644
* Tunnel : 0.368 → 0.489 → 0.542

Cong :
* Ref : 0.054 → 0.090 → 0.117
* UrbMask : 0.057 → 0.097 → 0.126
* Tunnel : 0.049 → 0.082 → 0.107

---

## Fig02 : délais de queue (p95/p99)

![](assets/final_figures_A_preview/Fig02_TailDelay_p95p99_Ret0Ret2.png)

NoCong : p95 (dist≤200 m) :
* Ref : 2.9 ms → 22.4 ms
* UrbMask : 3.9 ms → 22.9 ms
* Tunnel : 3.9 ms → 22.9 ms

Cong : p95 (dist≤200 m) :
* Ref : 69.6 ms → 79.9 ms
* UrbMask : 70.1 ms → 79.9 ms
* Tunnel : 69.1 ms → 79.6 ms

---

## Fig03 : Décomposition Cong-only (PHY / timely / late)

![](assets/final_figures_A_preview/Fig03_Cong_Decomposition_3Curves.png)

Lecture : bleu = \(\mathrm{PDR}_{phy}\), vert = \(\mathrm{PDR}_{timely}\), orange = \(\mathrm{late\_ratio}_{phy}\).  
Identité :
$$
\mathrm{PDR}*{\mathrm{timely}}=\mathrm{PDR}*{\mathrm{phy}}\cdot(1-\mathrm{late_ratio}_{\mathrm{phy}})
$$
Late% Fig07 :
* Ref : 5.5% → 8.7%
* UrbMask : 5.4% → 8.4%
* Tunnel : 5.1% → 7.9%

---

## Fig04 : Proxys de congestion

![](assets/final_figures_A_preview/Fig04_CongProxy_MeanStdBand.png)

Proxys proches entre scénarios (moyennes proches, variance faible) → congestion dominante.

---

## Fig05 : Hétérogénéité spatiale UrbMask

![](assets/final_figures_A_preview/Fig05_UrbMask_Heterogeneity_LinesAndRatio.png)

NoCong : PDR_band varie selon mid_x.  
Ratio Cong/NoCong : pénalisation plus forte sur segments faibles.

---

## Fig06 : Segmentation tunnel (inside vs outside)

![](assets/final_figures_A_preview/Fig06_Tunnel_InsideOutside_Bars_UnifiedY.png)

NoCong : inside < outside, ret améliore mais l’écart reste.  
Cong : inside proche de l’inutilisable, late apparaît ; cohérent avec Fig02/03.

---

## Fig07 : Matrices de synthèse

![](assets/final_figures_A_preview/Fig07_Summary_Matrices.png)

NoCong : gain PDR fort, coût de queue, late≈0.  
Cong : PDR faible dominée par contention, p95 proche deadline, late% augmente avec ret.

---

# Synthèse inter-figures

Déplacement du goulot d’étranglement, frontière gain–coût, dégradations structurées, implication : co‑conception ret + contrôle congestion.

---

# Limitations recommandées

(1) Résolution/smoothing : uniquement visualisation ; chiffres via Fig06/Fig07.  
(2) Quantiles : histogrammes ; masquage si échantillons insuffisants.  
(3) Proxys : indicateurs internes pour attribution ; pas reproduction full-stack.

## Aperçu de la structure du code

Le code est sous `03_sim_A/py/` et `03_sim_A/py/modules/`. Le pipeline est orchestré par `run_pipeline_A.py`. La documentation fichier par fichier est fournie dans le Notebook 2.